In [ ]:
import os

import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from dataset import *
from model import *
from trainer import Trainer
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import re

print("Program is started")
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
PATH = r"models/bert_classifier/"
print("Params is setted")
MAX_LEN = 200
BATCH_SIZE = 16

def clean_text(text):
    text = re.sub(r'http\S+', '', text)  # Удаляем ссылки
    text = re.sub(r'[^\w\s]', '', text)  # Удаляем специальные символы
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = text.lower()                  # Приводим к нижнему регистру
    return text


print("Data is starting to preparation")
train_data = pd.read_csv('../../data/input/tkk_train.csv')
test_data = pd.read_csv('../../data/input/tkk_etalon.csv')

train_data['text'] = train_data['text'].apply(lambda x: clean_text(x))
test_data['text'] = test_data['text'].apply(lambda x: clean_text(x))
#train_data = pd.read_csv(os.path.join(PATH, "train.csv"))
#test_data = pd.read_csv(os.path.join(PATH, "test.csv"))
print(train_data.head())
train_data.head()

le = LabelEncoder()

train_data.label = le.fit_transform(train_data.label)
test_data.label = le.fit_transform(test_data.label)
train_data.head()

train_split, val_split = train_test_split(train_data, test_size=0.85, random_state=42)

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/LaBSE", truncation=True, do_lower_case=True)

train_dataset = FiveDataset(train_split, tokenizer, MAX_LEN)
val_dataset = FiveDataset(val_split, tokenizer, MAX_LEN)
test_dataset = FiveDataset(test_data, tokenizer, MAX_LEN)

train_params = {"batch_size": BATCH_SIZE,
                "shuffle": True,
                "num_workers": 0
                }

test_params = {"batch_size": BATCH_SIZE,
               "shuffle": False,
               "num_workers": 0
               }

train_dataloader = DataLoader(train_dataset, **train_params)
val_dataloader = DataLoader(val_dataset, **test_params)
test_dataloader = DataLoader(test_dataset, **test_params)

config = {
    "num_classes": 3,
    "dropout_rate": 0.1
}
model = ModelForClassification(
    "sentence-transformers/LaBSE",
    config=config
)

trainer_config = {
    "lr": 2e-5,
    "n_epochs": 5,
    "weight_decay": 1e-6,
    "batch_size": BATCH_SIZE,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
}

t = Trainer(trainer_config)

t.fit(
    model,
    train_dataloader,
    val_dataloader
)

#t.save("baseline_model.ckpt")

t = Trainer.load("baseline_model.ckpt")
predictions = t.predict(test_dataloader)
ground_true = test_data['label'].tolist()
print("======Stats===========")
print(f1_score(ground_true, predictions, average='micro'))
print(f1_score(ground_true, predictions, average='macro'))
print(f1_score(ground_true, predictions, average='weighted'))
print(accuracy_score(ground_true, predictions))

print("======================")

# pred_labels = le.inverse_transform(preds)
cm = confusion_matrix(ground_true, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()


# sample_submission = pd.read_csv(r"C:\Users\ThinkPad T15 Gen1\OneDrive\Desktop\senti-analysis-sentrueval2016\models\bert_classifier\sample_submission.csv")
# sample_submission["rate"] = predictions
# sample_submission.rate = le.inverse_transform(sample_submission.rate)
# sample_submission.head()
#
# sample_submission.to_csv("submission.csv", index=False)

Program is started
cuda
Params is setted
Data is starting to preparation
   Unnamed: 0                                               text  label
0           0  mkomov максим вашем письмо мы получили наши со...      0
1           1            мегафон стал владельцем  акций евросети      0
2           2  rt fuckkiev evakobb мтс россия прислала жителя...     -1
3           3               видео  реклама со смехом мтс  супер       1
4           4  parfenov потому что мтс достало а пчел ненавиж...     -1
Epoch 1/5


  0%|          | 0/77 [00:00<?, ?it/s]

C:\Users\ThinkPad T15 Gen1\OneDrive\Desktop\ods_homework\nlp_huawei_new2_task\bert_venv\Lib\site-packages\transformers\tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


  0%|          | 0/437 [00:00<?, ?it/s]

0.7183603644371033
Epoch 2/5


...

0.782141387462616
Epoch 4/5


  0%|          | 0/77 [00:00<?, ?it/s]

  0%|          | 0/437 [00:00<?, ?it/s]

0.7636520266532898
Epoch 5/5


  0%|          | 0/77 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(ground_true, predictions))

In [ ]:
print("======Stats===========")
print(f1_score(ground_true, predictions, average='micro'))
print(f1_score(ground_true, predictions, average='macro'))
print(f1_score(ground_true, predictions, average='weighted'))
print(accuracy_score(ground_true, predictions))

print("======================")

======Stats===========
0.7179327521793275
0.6453892539695983
0.7271705394315245
0.7179327521793275
======================

======Stats===========
0.702729044834308
0.6268498664022232
0.705103772022147
0.702729044834308
======================

Banks - easy preprocessing
=====Stats===========
0.6924034869240349
0.5639816153007895
0.7043884593425526
0.6924034869240349
======================

TKK ruBERT- fitted with easy preprocessing
======Stats===========
0.6859785783836416
0.6083906357352044
0.68270462332796
0.6859785783836416
======================
Data with postprocessing tkk
======Stats===========
0.6820837390457644
0.5958337734331668
0.6805181186957397
0.6820837390457644
======================

In [ ]:
Data with postprocessing tkk
======Stats===========
0.6820837390457644
0.5958337734331668
0.6805181186957397
0.6820837390457644
======================